In [ ]:


import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as transforms
from torchvision.utils import save_image, make_grid
import json
import os
from datetime import datetime
import time
from tqdm import tqdm
import gc


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    torch.cuda.empty_cache()









Using device: cuda
GPU Memory: 17.2 GB


In [2]:
def load_processed_data():
    
    print("Loading processed data from baseline...")
    
    # Check if data exists
    data_dir = 'data/processed'
    required_files = ['X_train.npy', 'X_val.npy', 'X_test.npy', 
                     'y_train.npy', 'y_val.npy', 'y_test.npy', 'data_info.json']
    
    for file in required_files:
        filepath = os.path.join(data_dir, file)
        if not os.path.exists(filepath):
            raise FileNotFoundError(f"Required file not found: {filepath}\n"
                                  f"Please run the baseline notebook first and save the data!")
    
    # Load data
    X_train = np.load(f'{data_dir}/X_train.npy')
    X_val = np.load(f'{data_dir}/X_val.npy')
    X_test = np.load(f'{data_dir}/X_test.npy')
    y_train = np.load(f'{data_dir}/y_train.npy')
    y_val = np.load(f'{data_dir}/y_val.npy')
    y_test = np.load(f'{data_dir}/y_test.npy')
    
    # Load metadata
    with open(f'{data_dir}/data_info.json', 'r') as f:
        data_info = json.load(f)
    
    print("Data loaded successfully!")
    print(f"Train: {X_train.shape}")
    print(f"Val: {X_val.shape}")
    print(f"Test: {X_test.shape}")
    print(f"Image shape: {data_info['image_shape']}")
    print(f"Classes: {data_info['class_names']}")
    print(f"Minority classes: {data_info['minority_classes']}")
    
    return X_train, X_val, X_test, y_train, y_val, y_test, data_info

# Load the data
X_train, X_val, X_test, y_train, y_val, y_test, data_info = load_processed_data()

Loading processed data from baseline...
Data loaded successfully!
Train: (4800, 256, 256, 3)
Val: (1600, 256, 256, 3)
Test: (1600, 256, 256, 3)
Image shape: [256, 256, 3]
Classes: ['Disturbed Galaxies', 'Merging Galaxies', 'Round Smooth Galaxies', 'In-between Round Smooth Galaxies', 'Cigar Shaped Smooth Galaxies', 'Barred Spiral Galaxies', 'Unbarred Tight Spiral Galaxies', 'Unbarred Loose Spiral Galaxies', 'Edge-on Galaxies without Bulge', 'Edge-on Galaxies with Bulge']
Minority classes: [1, 4, 8, 9]


In [3]:
def cosine_beta_schedule(timesteps, s=0.008):
    
    steps = timesteps + 1
    x = torch.linspace(0, timesteps, steps)
    alphas_cumprod = torch.cos(((x / timesteps) + s) / (1 + s) * torch.pi * 0.5) ** 2
    alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
    betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
    return torch.clip(betas, 0.0001, 0.9999)

class SinusoidalPositionEmbeddings(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, time):
        device = time.device
        half_dim = self.dim // 2
        embeddings = np.log(10000) / (half_dim - 1)
        embeddings = torch.exp(torch.arange(half_dim, device=device) * -embeddings)
        embeddings = time[:, None] * embeddings[None, :]
        embeddings = torch.cat((embeddings.sin(), embeddings.cos()), dim=-1)
        return embeddings

class ConditionalBlock(nn.Module):
    
    def __init__(self, in_ch, out_ch, time_emb_dim, num_classes, dropout=0.1):
        super().__init__()
        self.time_mlp = nn.Linear(time_emb_dim, out_ch)
        self.class_emb = nn.Embedding(num_classes + 1, out_ch)  # +1 for unconditional
        
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        
        self.norm1 = nn.GroupNorm(8, out_ch)
        self.norm2 = nn.GroupNorm(8, out_ch)
        
        self.dropout = nn.Dropout(dropout)
        self.act = nn.SiLU()
        
        self.residual = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, time_emb, class_labels):
        h = self.conv1(x)
        h = self.norm1(h)
        
        # Add conditioning
        time_emb = self.time_mlp(time_emb)
        class_emb = self.class_emb(class_labels)
        cond = (time_emb + class_emb).unsqueeze(-1).unsqueeze(-1)
        h = h + cond
        
        h = self.act(h)
        h = self.dropout(h)
        
        h = self.conv2(h)
        h = self.norm2(h)
        h = self.act(h)
        
        return h + self.residual(x)

class ConditionalUNet(nn.Module):
    
    def __init__(self, in_channels=3, out_channels=3, num_classes=4, time_emb_dim=128):
        super().__init__()
        
        # Time embedding
        self.time_mlp = nn.Sequential(
            SinusoidalPositionEmbeddings(time_emb_dim),
            nn.Linear(time_emb_dim, time_emb_dim * 2),
            nn.SiLU(),
            nn.Linear(time_emb_dim * 2, time_emb_dim),
        )
        
        # Encoder
        self.down1 = ConditionalBlock(in_channels, 64, time_emb_dim, num_classes)
        self.down2 = ConditionalBlock(64, 128, time_emb_dim, num_classes)
        self.down3 = ConditionalBlock(128, 256, time_emb_dim, num_classes)
        
        # Middle
        self.mid = ConditionalBlock(256, 256, time_emb_dim, num_classes)
        
        # Decoder
        self.up3 = ConditionalBlock(512, 128, time_emb_dim, num_classes)  # 256 + 256 skip
        self.up2 = ConditionalBlock(256, 64, time_emb_dim, num_classes)   # 128 + 128 skip
        self.up1 = ConditionalBlock(128, 64, time_emb_dim, num_classes)   # 64 + 64 skip
        
        self.out = nn.Conv2d(64, out_channels, 1)
        self.pool = nn.MaxPool2d(2)
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        
    def forward(self, x, timestep, class_labels):
        time_emb = self.time_mlp(timestep)
        
        # Encoder with skip connections
        skip1 = self.down1(x, time_emb, class_labels)
        x = self.pool(skip1)
        
        skip2 = self.down2(x, time_emb, class_labels)
        x = self.pool(skip2)
        
        skip3 = self.down3(x, time_emb, class_labels)
        x = self.pool(skip3)
        
        # Middle
        x = self.mid(x, time_emb, class_labels)
        
        # Decoder with skip connections
        x = self.upsample(x)
        x = torch.cat([x, skip3], dim=1)
        x = self.up3(x, time_emb, class_labels)
        
        x = self.upsample(x)
        x = torch.cat([x, skip2], dim=1)
        x = self.up2(x, time_emb, class_labels)
        
        x = self.upsample(x)
        x = torch.cat([x, skip1], dim=1)
        x = self.up1(x, time_emb, class_labels)
        
        return self.out(x)

class DDPM:
    
    def __init__(self, model, timesteps=1000):
        self.model = model
        self.timesteps = timesteps
        
        # Beta schedule
        self.betas = cosine_beta_schedule(timesteps)
        self.alphas = 1.0 - self.betas
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0)
        
        # Pre-compute values for training and sampling
        self.sqrt_alphas_cumprod = torch.sqrt(self.alphas_cumprod)
        self.sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - self.alphas_cumprod)
        
        # For sampling
        self.posterior_variance = self.betas * (1.0 - F.pad(self.alphas_cumprod[:-1], (1, 0), value=1.0)) / (1.0 - self.alphas_cumprod)
        
    def q_sample(self, x_start, t, noise=None):
        
        if noise is None:
            noise = torch.randn_like(x_start)
            
        sqrt_alphas_cumprod_t = self.sqrt_alphas_cumprod[t].view(-1, 1, 1, 1)
        sqrt_one_minus_alphas_cumprod_t = self.sqrt_one_minus_alphas_cumprod[t].view(-1, 1, 1, 1)
        
        return sqrt_alphas_cumprod_t * x_start + sqrt_one_minus_alphas_cumprod_t * noise
    
    def p_losses(self, x_start, t, class_labels, noise=None):
        
        if noise is None:
            noise = torch.randn_like(x_start)
            
        x_noisy = self.q_sample(x_start=x_start, t=t, noise=noise)
        predicted_noise = self.model(x_noisy, t, class_labels)
        
        return F.mse_loss(noise, predicted_noise)
    
    @torch.no_grad()
    def sample(self, shape, class_labels):
        
        device = next(self.model.parameters()).device
        b = shape[0]
        
        # Start from pure noise
        img = torch.randn(shape, device=device)
        
        for i in tqdm(reversed(range(0, self.timesteps)), desc='Sampling'):
            t = torch.full((b,), i, device=device, dtype=torch.long)
            
            betas_t = self.betas[t].view(-1, 1, 1, 1)
            sqrt_one_minus_alphas_cumprod_t = self.sqrt_one_minus_alphas_cumprod[t].view(-1, 1, 1, 1)
            sqrt_recip_alphas_t = torch.sqrt(1.0 / self.alphas[t]).view(-1, 1, 1, 1)
            
            # Predict noise and remove it
            model_mean = sqrt_recip_alphas_t * (
                img - betas_t * self.model(img, t, class_labels) / sqrt_one_minus_alphas_cumprod_t
            )
            
            if i > 0:
                posterior_variance_t = self.posterior_variance[t].view(-1, 1, 1, 1)
                noise = torch.randn_like(img)
                img = model_mean + torch.sqrt(posterior_variance_t) * noise
            else:
                img = model_mean
                
        return img

class Galaxy10DiffusionDataset(Dataset):
    
    
    def __init__(self, images, labels, focus_classes=None, transform=None):
        # Convert to diffusion format [-1, 1]
        if images.dtype == np.uint8:
            self.images = (images.astype(np.float32) / 127.5) - 1.0
        else:
            self.images = images
            
        self.labels = labels.astype(np.int64)
        self.transform = transform
        
        # Filter to focus classes and remap labels
        if focus_classes is not None:
            mask = np.isin(labels, focus_classes)
            self.images = self.images[mask]
            self.labels = self.labels[mask]
            
            # Create label mapping: old class -> new index
            self.label_mapping = {old: new for new, old in enumerate(focus_classes)}
            self.labels = np.array([self.label_mapping[label] for label in self.labels])
            
            print(f"Filtered to {len(self.images)} samples from classes {focus_classes}")
            print(f"Label mapping: {self.label_mapping}")
        
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        image = self.images[idx]
        label = self.labels[idx]
        
        if self.transform:
            from PIL import Image
            # Convert to PIL for transforms
            image_pil = Image.fromarray(((image + 1.0) * 127.5).astype(np.uint8))
            image = self.transform(image_pil)
            # Back to [-1, 1]
            image = (image - 0.5) * 2.0
        else:
            image = torch.from_numpy(image).permute(2, 0, 1)
        
        return image, torch.tensor(label, dtype=torch.long)

In [4]:
def train_diffusion_model():
  
    print("TRAINING CONDITIONAL DIFFUSION MODEL")
   
    
    # Training parameters
    BATCH_SIZE = 4
    EPOCHS = 20  
    LEARNING_RATE = 1e-4
    TIMESTEPS = 1000
    
   
    minority_classes = [1, 4, 8, 9] 
   
    
    print(f"Training parameters:")
    print(f"- Batch size: {BATCH_SIZE}")
    print(f"- Epochs: {EPOCHS}")
    print(f"- Learning rate: {LEARNING_RATE}")
    print(f"- Timesteps: {TIMESTEPS}")
    print(f"- Focus classes: {minority_classes}")
    
    
    train_transform = transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(15),
        transforms.ToTensor()
    ])
    
    val_transform = transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.ToTensor()
    ])
    
    
    train_dataset = Galaxy10DiffusionDataset(
        X_train, y_train, 
        focus_classes=minority_classes,
        transform=train_transform
    )
    
    val_dataset = Galaxy10DiffusionDataset(
        X_val, y_val,
        focus_classes=minority_classes,
        transform=val_transform
    )
    
    # Data loaders
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    
    
    model = ConditionalUNet(num_classes=len(minority_classes)).to(device)
    ddpm = DDPM(model, timesteps=TIMESTEPS)
    
   
    ddpm.betas = ddpm.betas.to(device)
    ddpm.alphas = ddpm.alphas.to(device)
    ddpm.alphas_cumprod = ddpm.alphas_cumprod.to(device)
    ddpm.sqrt_alphas_cumprod = ddpm.sqrt_alphas_cumprod.to(device)
    ddpm.sqrt_one_minus_alphas_cumprod = ddpm.sqrt_one_minus_alphas_cumprod.to(device)
    ddpm.posterior_variance = ddpm.posterior_variance.to(device)
    
    # Optimizer
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    
    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
    
    # Training loop
    train_losses = []
    val_losses = []
    best_val_loss = float('inf')
    
    
    os.makedirs('results/diffusion', exist_ok=True)
    
    start_time = time.time()
    
    for epoch in range(EPOCHS):
        print(f"\nEpoch {epoch+1}/{EPOCHS}")
      
        
        # Training
        model.train()
        epoch_train_loss = 0
        num_batches = 0
        
        for images, labels in tqdm(train_loader, desc="Training"):
            images, labels = images.to(device), labels.to(device)
            
            # Random timesteps
            t = torch.randint(0, TIMESTEPS, (images.shape[0],), device=device).long()
            
            # Calculate loss
            loss = ddpm.p_losses(images, t, labels)
            
            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            epoch_train_loss += loss.item()
            num_batches += 1
        
        avg_train_loss = epoch_train_loss / num_batches
        train_losses.append(avg_train_loss)
        
        # Validation
        model.eval()
        epoch_val_loss = 0
        num_val_batches = 0
        
        with torch.no_grad():
            for images, labels in tqdm(val_loader, desc="Validation"):
                images, labels = images.to(device), labels.to(device)
                t = torch.randint(0, TIMESTEPS, (images.shape[0],), device=device).long()
                loss = ddpm.p_losses(images, t, labels)
                epoch_val_loss += loss.item()
                num_val_batches += 1
        
        avg_val_loss = epoch_val_loss / num_val_batches if num_val_batches > 0 else float('inf')
        val_losses.append(avg_val_loss)
        
        scheduler.step()
        
        print(f"Train Loss: {avg_train_loss:.4f}")
        print(f"Val Loss: {avg_val_loss:.4f}")
        
        # Save best model
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save({
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'epoch': epoch,
                'train_loss': avg_train_loss,
                'val_loss': avg_val_loss,
                'minority_classes': minority_classes
            }, 'results/diffusion/best_model.pth')
            print(f"Saved best model!")
        
        # Generate samples every 5 epochs
        if (epoch + 1) % 5 == 0:
            generate_samples(ddpm, epoch + 1, minority_classes)
        
        torch.cuda.empty_cache()
    
    training_time = time.time() - start_time
    print(f"\nTraining completed in {training_time/60:.1f} minutes")
    
    # Save results
    results = {
        'train_losses': train_losses,
        'val_losses': val_losses,
        'best_val_loss': best_val_loss,
        'training_time_minutes': training_time / 60,
        'config': {
            'epochs': EPOCHS,
            'batch_size': BATCH_SIZE,
            'learning_rate': LEARNING_RATE,
            'timesteps': TIMESTEPS,
            'minority_classes': minority_classes
        }
    }
    
    with open('results/diffusion/training_results.json', 'w') as f:
        json.dump(results, f, indent=2)
    
    return ddpm, results

In [5]:
def generate_samples(ddpm, epoch, minority_classes, num_samples=8):
   
    ddpm.model.eval()
    os.makedirs('results/diffusion/samples', exist_ok=True)
    
    with torch.no_grad():
        for class_idx, original_class in enumerate(minority_classes):
            # Generate samples for this class
            class_labels = torch.full((num_samples,), class_idx, device=device, dtype=torch.long)
            
            samples = ddpm.sample(
                shape=(num_samples, 3, 128, 128),
                class_labels=class_labels
            )
            
            # Convert from [-1, 1] to [0, 1]
            samples = (samples + 1) / 2
            samples = torch.clamp(samples, 0, 1)
            
            # Save grid
            grid = make_grid(samples, nrow=4, padding=2)
            save_image(grid, f'results/diffusion/samples/epoch_{epoch}_class_{original_class}.png')

In [6]:
ddpm, results = train_diffusion_model()

TRAINING CONDITIONAL DIFFUSION MODEL
Training parameters:
- Batch size: 4
- Epochs: 20
- Learning rate: 0.0001
- Timesteps: 1000
- Focus classes: [1, 4, 8, 9]
Filtered to 1486 samples from classes [1, 4, 8, 9]
Label mapping: {1: 0, 4: 1, 8: 2, 9: 3}
Filtered to 495 samples from classes [1, 4, 8, 9]
Label mapping: {1: 0, 4: 1, 8: 2, 9: 3}
Model parameters: 3,688,835

Epoch 1/20


Validation: 100%|██████████| 124/124 [00:03<00:00, 34.48it/s]


Train Loss: 0.1559
Val Loss: 0.0658
Saved best model!

Epoch 2/20


Validation: 100%|██████████| 124/124 [00:03<00:00, 34.80it/s]


Train Loss: 0.0686
Val Loss: 0.0480
Saved best model!

Epoch 3/20


Validation: 100%|██████████| 124/124 [00:03<00:00, 34.44it/s]


... (39 outputs left out)

Validation: 100%|██████████| 124/124 [00:03<00:00, 35.12it/s]


Train Loss: 0.0508
Val Loss: 0.0405


Sampling: 1000it [00:39, 25.42it/s]
Sampling: 1000it [00:39, 25.41it/s]
Sampling: 1000it [00:39, 25.20it/s]
Sampling: 1000it [00:40, 24.99it/s]



Training completed in 21.4 minutes


In [7]:
def generate_synthetic_dataset(ddpm, minority_classes, samples_per_class=200, guidance_scale=0.0):
    
    print(f"\nGenerating synthetic dataset:")
    print(f"- Classes: {minority_classes}")
    print(f"- Samples per class: {samples_per_class}")
    
    device = next(ddpm.model.parameters()).device  # Now matches your variable name
    ddpm.model.eval()
    
    synthetic_images = []
    synthetic_labels = []
    
    with torch.no_grad():
        for class_idx, class_id in enumerate(minority_classes):
            print(f"Generating samples for class {class_id}...")
            
            
            batch_size = 8
            num_batches = (samples_per_class + batch_size - 1) // batch_size
            
            class_samples = []
            
            for batch in range(num_batches):
                current_batch_size = min(batch_size, samples_per_class - batch * batch_size)
                if current_batch_size <= 0:
                    break
                
                
                class_labels = torch.full((current_batch_size,), class_idx, device=device, dtype=torch.long)
                
                # Generate samples
                samples = ddpm.sample(  
                    shape=(current_batch_size, 3, 128, 128),
                    class_labels=class_labels
                )
                
                # Convert from [-1, 1] to [0, 1]
                samples = (samples + 1) / 2
                samples = torch.clamp(samples, 0, 1)
                
                # Convert to numpy
                samples_np = samples.cpu().numpy()
                class_samples.append(samples_np)
                
                # Create labels
                batch_labels = np.full(current_batch_size, class_id)
                synthetic_labels.extend(batch_labels)
                
                torch.cuda.empty_cache()
            
            if class_samples:
                class_samples = np.concatenate(class_samples, axis=0)
                synthetic_images.append(class_samples)
    
    # Concatenate all classes
    synthetic_images = np.concatenate(synthetic_images, axis=0)
    synthetic_labels = np.array(synthetic_labels)
    
    # Convert to uint8 format (0-255) for consistency
    synthetic_images = (synthetic_images * 255).astype(np.uint8)
    # Transpose from (N, C, H, W) to (N, H, W, C)
    synthetic_images = np.transpose(synthetic_images, (0, 2, 3, 1))
    
    print(f"Generated synthetic dataset:")
    print(f"- Shape: {synthetic_images.shape}")
    print(f"- Labels: {len(synthetic_labels)}")
    print(f"- Classes: {np.unique(synthetic_labels)}")
    
   
    np.save('results/diffusion/synthetic_images.npy', synthetic_images)
    np.save('results/diffusion/synthetic_labels.npy', synthetic_labels)
    
    return synthetic_images, synthetic_labels

In [11]:
# Save the ddpm object for reuse
torch.save({
    'ddpm_state': ddpm,  
    'minority_classes': [1, 4, 8, 9],
    'label_mapping': {0: 0, 1: 1, 4: 2, 7: 3}
}, 'results/diffusion/complete_diffusion_model.pth')

In [12]:
# Generate synthetic samples for augmentation
synthetic_images, synthetic_labels = generate_synthetic_dataset(
    ddpm, 
    minority_classes=[1, 4, 8, 9],
    samples_per_class=200  # Generate 200 per minority class
)


Generating synthetic dataset:
- Classes: [1, 4, 8, 9]
- Samples per class: 200
Generating samples for class 1...


Sampling: 1000it [00:39, 25.01it/s]
Sampling: 1000it [00:39, 25.10it/s]
Sampling: 1000it [00:39, 25.06it/s]
Sampling: 1000it [00:39, 25.11it/s]
Sampling: 1000it [00:39, 25.07it/s]
Sampling: 1000it [00:40, 24.78it/s]
Sampling: 1000it [00:39, 25.41it/s]
Sampling: 1000it [00:39, 25.41it/s]
Sampling: 1000it [00:39, 25.40it/s]
Sampling: 1000it [00:39, 25.40it/s]
Sampling: 1000it [00:39, 25.39it/s]
Sampling: 1000it [00:39, 25.41it/s]
Sampling: 1000it [00:39, 25.41it/s]
Sampling: 1000it [00:39, 25.40it/s]
Sampling: 1000it [00:39, 25.42it/s]
Sampling: 1000it [00:39, 25.41it/s]
Sampling: 1000it [00:39, 25.40it/s]
Sampling: 1000it [00:39, 25.41it/s]
Sampling: 1000it [00:39, 25.41it/s]
Sampling: 1000it [00:39, 25.41it/s]
Sampling: 1000it [00:39, 25.40it/s]
Sampling: 1000it [00:39, 25.42it/s]
Sampling: 1000it [00:39, 25.41it/s]
Sampling: 1000it [00:40, 24.91it/s]
Sampling: 1000it [00:39, 25.14it/s]


Generating samples for class 4...


Sampling: 1000it [00:40, 25.00it/s]
Sampling: 1000it [00:39, 25.05it/s]
Sampling: 1000it [00:40, 24.94it/s]
Sampling: 1000it [00:40, 24.98it/s]
Sampling: 1000it [00:40, 24.98it/s]
Sampling: 1000it [00:40, 24.97it/s]
Sampling: 1000it [00:39, 25.05it/s]
Sampling: 1000it [00:39, 25.06it/s]
Sampling: 1000it [00:39, 25.05it/s]
Sampling: 1000it [00:39, 25.39it/s]
Sampling: 1000it [00:39, 25.40it/s]
Sampling: 1000it [00:39, 25.40it/s]
Sampling: 1000it [00:39, 25.41it/s]
Sampling: 1000it [00:39, 25.05it/s]
Sampling: 1000it [00:39, 25.04it/s]
Sampling: 1000it [00:39, 25.04it/s]
Sampling: 1000it [00:39, 25.04it/s]
Sampling: 1000it [00:39, 25.04it/s]
Sampling: 1000it [00:39, 25.04it/s]
Sampling: 1000it [00:39, 25.02it/s]
Sampling: 1000it [00:39, 25.02it/s]
Sampling: 1000it [00:39, 25.01it/s]
Sampling: 1000it [00:39, 25.01it/s]
Sampling: 1000it [00:39, 25.01it/s]
Sampling: 1000it [00:39, 25.02it/s]


Generating samples for class 8...


Sampling: 1000it [00:39, 25.03it/s]
Sampling: 1000it [00:39, 25.04it/s]
Sampling: 1000it [00:39, 25.03it/s]
Sampling: 1000it [00:39, 25.03it/s]
Sampling: 1000it [00:39, 25.04it/s]
Sampling: 1000it [00:39, 25.02it/s]
Sampling: 1000it [00:39, 25.01it/s]
Sampling: 1000it [00:39, 25.01it/s]
Sampling: 1000it [00:39, 25.01it/s]
Sampling: 1000it [00:39, 25.01it/s]
Sampling: 1000it [00:39, 25.01it/s]
Sampling: 1000it [00:39, 25.01it/s]
Sampling: 1000it [00:39, 25.01it/s]
Sampling: 1000it [00:39, 25.00it/s]
Sampling: 1000it [00:39, 25.00it/s]
Sampling: 1000it [00:39, 25.01it/s]
Sampling: 1000it [00:39, 25.01it/s]
Sampling: 1000it [00:40, 24.96it/s]
Sampling: 1000it [00:40, 24.96it/s]
Sampling: 1000it [00:40, 24.96it/s]
Sampling: 1000it [00:40, 24.96it/s]
Sampling: 1000it [00:40, 24.96it/s]
Sampling: 1000it [00:40, 24.95it/s]
Sampling: 1000it [00:39, 25.03it/s]
Sampling: 1000it [00:39, 25.04it/s]


Generating samples for class 9...


Sampling: 1000it [00:39, 25.41it/s]
Sampling: 1000it [00:39, 25.41it/s]
Sampling: 1000it [00:39, 25.41it/s]
Sampling: 1000it [00:39, 25.37it/s]
Sampling: 1000it [00:39, 25.02it/s]
Sampling: 1000it [00:39, 25.01it/s]
Sampling: 1000it [00:39, 25.01it/s]
Sampling: 1000it [00:39, 25.02it/s]
Sampling: 1000it [00:39, 25.01it/s]
Sampling: 1000it [00:39, 25.00it/s]
Sampling: 1000it [00:39, 25.01it/s]
Sampling: 1000it [00:39, 25.01it/s]
Sampling: 1000it [00:39, 25.01it/s]
Sampling: 1000it [00:39, 25.01it/s]
Sampling: 1000it [00:40, 24.97it/s]
Sampling: 1000it [00:40, 24.97it/s]
Sampling: 1000it [00:40, 24.96it/s]
Sampling: 1000it [00:40, 24.97it/s]
Sampling: 1000it [00:40, 24.97it/s]
Sampling: 1000it [00:40, 24.96it/s]
Sampling: 1000it [00:40, 24.96it/s]
Sampling: 1000it [00:39, 25.03it/s]
Sampling: 1000it [00:39, 25.04it/s]
Sampling: 1000it [00:39, 25.04it/s]
Sampling: 1000it [00:39, 25.04it/s]


Generated synthetic dataset:
- Shape: (800, 128, 128, 3)
- Labels: 800
- Classes: [1 4 8 9]
